# Local Voice Agent — Gemma 4 + LiveKit + Whisper + Qwen3-TTS

A fully local, real-time voice assistant that runs on a Kaggle GPU notebook:

| Layer | Tool | Role |
|---|---|---|
| Transport / orchestration | **LiveKit Agents** | connects your mic to the pipeline below, handles turn-taking |
| Ears (STT) | **faster-whisper** | your voice → text |
| Brain (LLM) | **Gemma 4** via **Ollama** | text → text reply |
| Mouth (TTS) | **Qwen3-TTS** | text → voice |

> **Before you run this**, add three Kaggle secrets: `LIVEKIT_URL`, `LIVEKIT_API_KEY`,
> `LIVEKIT_API_SECRET` (Add-ons → Secrets). Turn the GPU accelerator ON.
>
> **If a cell throws `worker is already running` or `address already in use`**,
> see `ERRORS_AND_SOLUTIONS.md` — most of the time the fix is either re-running
> the cell, or doing Kernel → Restart & clear the stuck background process.


In [ ]:
# ---- What each package does -------------------------------------------------
# livekit-agents  : the framework that connects your microphone to our AI models
#     [silero]    :   -> bundled voice-activity detection (knows when you start/stop talking)
#     [openai]    :   -> lets us talk to any "OpenAI-style" model endpoint (Ollama exposes one)
# faster-whisper  : Speech-to-Text  (your voice  -> text)
# qwen-tts        : Text-to-Speech  (AI's reply  -> voice)
# soundfile       : a small helper for handling audio data
#
# ---- Why the extra apt-get packages are needed (see ERRORS_AND_SOLUTIONS.md) -
# zstd            : the Ollama installer ships its binaries zstd-compressed;
#                   without this, install.sh fails with "requires zstd" (Error 1)
# pciutils/lshw   : hardware-probing tools Ollama uses to detect the GPU inside
#                   this container (Error 2)
# sox/libsox-...  : OS-level audio codec library required by faster-whisper /
#                   voice-activity-detection to decode raw audio streams (Error 3)
# ------------------------------------------------------------------------------
!pip install -q \
    "livekit-agents[silero,openai]~=1.6" \
    "faster-whisper>=1.2" \
    "qwen-tts>=0.1.1" \
    soundfile

!sudo apt-get install -y zstd pciutils lshw sox libsox-fmt-all

# Ollama is a small program that runs the Gemma 4 model locally and exposes it
# through an OpenAI-compatible HTTP API. This line installs the Ollama server.
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# Read the three secrets you saved in Kaggle and put them in environment
# variables, where LiveKit knows to look for them.
secrets = UserSecretsClient()
os.environ["LIVEKIT_URL"]        = secrets.get_secret("LIVEKIT_URL")         # wss://<your-project>.livekit.cloud
os.environ["LIVEKIT_API_KEY"]    = secrets.get_secret("LIVEKIT_API_KEY")
os.environ["LIVEKIT_API_SECRET"] = secrets.get_secret("LIVEKIT_API_SECRET")

print("LiveKit credentials loaded ✔")


In [ ]:
import socket

# ---- Re-run safety net -------------------------------------------------------
# Jupyter/Kaggle notebooks keep background processes (Ollama server, LiveKit
# worker, health-check server) alive even after a cell finishes or errors out.
# Re-running a "start the server" cell a second time then crashes with
# "address already in use" (Errors #4 and #5). This helper lets every launch
# cell below check first and skip re-launching if something is already up.
# -------------------------------------------------------------------------------
def is_port_in_use(port: int, host: str = "127.0.0.1") -> bool:
    """Return True if something is already listening on host:port."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        return sock.connect_ex((host, port)) == 0

# If you ever get a *fatal* "address already in use" crash that this
# skip-logic doesn't resolve (e.g. a truly stuck/zombie process from a crashed
# run), force-kill whatever is bound to the port and re-run the cell:
#     !fuser -k 11434/tcp   # Ollama's default port
#     !fuser -k 8081/tcp    # LiveKit's internal health-check port
print("Port-check helper ready ✔")


In [ ]:
import subprocess, time, requests

OLLAMA_PORT = 11434

# ---- Speed up token generation / reduce VRAM use (see Error #8) -------------
# Flash Attention is disabled in Ollama by default. It must be set BEFORE the
# server process starts, or it has no effect.
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"

# 1) Start the Ollama server in the background (skip if a previous run of this
#    cell already has it listening — see the port-check helper above).
if is_port_in_use(OLLAMA_PORT):
    print(f"Ollama already running on port {OLLAMA_PORT} — skipping re-launch.")
else:
    # Containers don't run systemd, so Ollama can't register itself as a
    # background service the normal way (Error #2). We launch it ourselves
    # with subprocess.Popen() instead, which keeps it running in the
    # background without blocking this cell.
    subprocess.Popen(["ollama", "serve"])

    # 2) Wait until the server is actually ready before we ask it for anything.
    print("Waiting for Ollama to start...")
    for _ in range(30):
        try:
            if requests.get(f"http://localhost:{OLLAMA_PORT}", timeout=2).ok:
                print("Ollama is up ✔")
                break
        except Exception:
            pass          # not ready yet, keep waiting
        time.sleep(2)

# 3) Download Gemma 4. We save the exact model name in a variable so we can
#    reuse it later when we tell the assistant which "brain" to use.
#    NOTE: this is deliberately its own cell/step, run *before* the live
#    session starts. Downloading a multi-gigabyte model *during* a live
#    LiveKit connection blocks the Python thread long enough that LiveKit
#    thinks the agent has crashed and drops the call (Error #6).
GEMMA_MODEL = "gemma4:e4b-it-qat"
print(f"Downloading {GEMMA_MODEL} (first time only, please wait)...")
subprocess.run(["ollama", "pull", GEMMA_MODEL], check=True)
print("Gemma 4 is ready ✔")


In [ ]:
import asyncio
import torch
import numpy as np
from qwen_tts import Qwen3TTSModel
from livekit.agents import tts, DEFAULT_API_CONNECT_OPTIONS

# Pick one of Qwen's built-in voices.
# Try others too: Eric, Aiden, Vivian, Serena, Dylan, Ryan, ...
TTS_SPEAKER  = "Ryan"
TTS_LANGUAGE = "English"


class QwenTTS(tts.TTS):
    """A LiveKit voice engine powered by the local Qwen3-TTS model."""

    def __init__(self):
        # Describe our audio to LiveKit. Qwen3-TTS makes a whole sentence at
        # once (it does not stream word-by-word), so streaming=False.
        super().__init__(
            capabilities=tts.TTSCapabilities(streaming=False),
            sample_rate=24000,   # a sensible default; we use the model's real rate below
            num_channels=1,      # mono audio (one channel)
        )
        # Load the 0.6B voice model onto the GPU.
        # "sdpa" attention works on every GPU; only switch to
        # "flash_attention_2" if you have installed the flash-attn package.
        self._model = Qwen3TTSModel.from_pretrained(
            "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice",
            device_map="cuda:0",
            dtype=torch.bfloat16,
            attn_implementation="sdpa",
        )

    # LiveKit calls this each time the assistant wants to say something.
    def synthesize(self, text, *, conn_options=DEFAULT_API_CONNECT_OPTIONS):
        return _QwenStream(tts=self, input_text=text, conn_options=conn_options)


class _QwenStream(tts.ChunkedStream):
    """Turns one piece of text into audio and sends it back to LiveKit."""

    async def _run(self, output_emitter):
        model = self._tts._model

        # Generating audio on the GPU is "blocking" work, so we run it in a
        # background thread. That keeps the assistant responsive.
        def _generate():
            wavs, sample_rate = model.generate_custom_voice(
                text=self._input_text,
                language=TTS_LANGUAGE,
                speaker=TTS_SPEAKER,
            )
            # Convert the raw waveform (floats between -1.0 and 1.0) into
            # 16-bit PCM bytes, the format LiveKit sends over the network.
            audio = np.asarray(wavs[0], dtype=np.float32)
            pcm16 = (np.clip(audio, -1.0, 1.0) * 32767).astype(np.int16).tobytes()
            return pcm16, int(sample_rate)

        pcm16, sample_rate = await asyncio.to_thread(_generate)

        # Hand the finished audio to LiveKit so it can play it for the user.
        output_emitter.initialize(
            request_id="qwen3-tts",
            sample_rate=sample_rate,
            num_channels=1,
            mime_type="audio/pcm",
        )
        output_emitter.push(pcm16)
        output_emitter.flush()


print("Text-to-Speech (Qwen3-TTS) wrapper ready ✔")


In [ ]:
from livekit.agents import stt
from livekit import rtc
from faster_whisper import WhisperModel


class LocalWhisperSTT(stt.STT):
    """A LiveKit listening engine powered by the local Whisper model."""

    def __init__(self, model_size="small"):
        # Whisper here transcribes a full sentence after you stop talking,
        # so it is not a streaming/word-by-word engine.
        super().__init__(
            capabilities=stt.STTCapabilities(streaming=False, interim_results=False),
        )
        # Load Whisper onto the GPU. "small" is a good speed/accuracy balance
        # on a T4. Other sizes: "base", "medium", "large-v3", "turbo".
        self._model = WhisperModel(model_size, device="cuda", compute_type="float16")

    # LiveKit calls this with your recorded audio once you finish speaking.
    async def _recognize_impl(self, buffer, *, language=None, conn_options=None):
        # LiveKit may pass a special "not given" value for language; only keep
        # it if it is a real language string (otherwise let Whisper auto-detect).
        lang = language if isinstance(language, str) else None

        # Combine the audio pieces into one, then convert the 16-bit samples
        # into the float32 format that Whisper expects.
        frame = rtc.combine_audio_frames(buffer)
        samples = np.frombuffer(frame.data, dtype=np.int16).astype(np.float32) / 32768.0

        # Transcribing is blocking work, so run it in a background thread.
        def _transcribe():
            segments, _info = self._model.transcribe(samples, language=lang)
            return " ".join(seg.text for seg in segments).strip()

        text = await asyncio.to_thread(_transcribe)

        # Return the recognized text in the shape LiveKit expects.
        return stt.SpeechEvent(
            type=stt.SpeechEventType.FINAL_TRANSCRIPT,
            alternatives=[stt.SpeechData(language=lang or "en", text=text)],
        )


print("Speech-to-Text (Whisper) wrapper ready ✔")


In [ ]:
# ---- Pre-warm the model so the first reply isn't slow (see Error #7) --------
# The *very first* request to Ollama has to physically move Gemma 4 from disk
# into GPU VRAM. That "cold start" takes ~15 seconds — long enough that when
# LiveKit asks the agent to say hello, it exceeds LiveKit's default patience
# and throws an APITimeoutError before the model ever finishes loading.
#
# Sending one silent, throwaway request here — *before* the live session
# starts — forces the model into VRAM ahead of time, so it's instantly ready
# the moment a real caller connects.
print("Pre-warming Gemma 4 (loading it into GPU memory)...")
requests.post(
    f"http://localhost:{OLLAMA_PORT}/api/generate",
    json={"model": GEMMA_MODEL, "prompt": "Hello", "stream": False},
    timeout=60,
)
print("Gemma 4 is warm and ready ✔")


In [ ]:
import nest_asyncio
# Notebooks already run an asyncio event loop under the hood. LiveKit's SDK
# wants to run its own loop too, and also keeps a strict "only one worker per
# process" registry that a stopped cell does not clear (Error #10). Patching
# the loop with nest_asyncio avoids event-loop conflicts; if you still hit
# "worker is already running", do Kernel -> Restart & Run All.
nest_asyncio.apply()

from livekit.agents import Agent, AgentSession, JobContext, AgentServer, JobExecutorType
from livekit.plugins import openai  # NOTE: silero is intentionally NOT imported here —
                                     # AgentSession now bundles VAD automatically, so the
                                     # old `from livekit.plugins import silero` +
                                     # `vad=silero.VAD.load()` pattern is deprecated (Error #9).

# Create the worker.
#  - THREAD executor: run the agent INSIDE this notebook so it reuses the GPU
#    models we already loaded (instead of starting a separate process).
#  - load_threshold=inf: always accept calls, even if the machine looks busy
#    while models are loading (avoids "the agent never joins" confusion).
#  - port=8082: LiveKit's internal health-check server defaults to 8081, which
#    a previous crashed run may have left occupied. Using a different port
#    permanently sidesteps that "address already in use" crash (Error #5).
server = AgentServer(
    job_executor_type=JobExecutorType.THREAD,
    load_threshold=float("inf"),
    port=8082,
    ws_url=os.environ["LIVEKIT_URL"],
    api_key=os.environ["LIVEKIT_API_KEY"],
    api_secret=os.environ["LIVEKIT_API_SECRET"],
)


# This function runs once for every person who joins a room.
# `agent_name` must match the identity the LiveKit Sandbox/Console explicitly
# dispatches to — an empty decorator defaults to auto-dispatch and the agent
# never shows up in the Sandbox's participant list (Error #11).
@server.rtc_session(agent_name="A_Ki2SWSVusb6t")
async def entrypoint(ctx: JobContext):
    # Connect this worker to the room the user just joined.
    await ctx.connect()

    # Build the conversation pipeline.
    session = AgentSession(
        # Voice-activity detection is now handled automatically under the hood.
        turn_detection="vad",           # uses that to know when it's your turn
        stt=LocalWhisperSTT("small"),   # ears  : your voice -> text
        llm=openai.LLM.with_ollama(     # brain : Gemma 4, served locally by Ollama
            model=GEMMA_MODEL,
            base_url=f"http://localhost:{OLLAMA_PORT}/v1",
            # NOTE: don't pass timeout= here. with_ollama() is a thin wrapper
            # and doesn't accept raw HTTP client kwargs like `timeout`
            # (Error #12) — the pre-warm step above already solves the
            # slow-first-reply problem this would have tried to work around.
        ),
        tts=QwenTTS(),                  # mouth : text -> voice
    )

    # Give the assistant its personality.
    agent = Agent(
        instructions="You are a concise, friendly voice assistant powered by Gemma 4."
    )

    # Start listening in the room.
    await session.start(agent=agent, room=ctx.room)

    # Say hello first, so the user knows it is working.
    await session.generate_reply(
        instructions="Greet the user warmly and briefly ask how you can help."
    )


print("Assistant assembled ✔")


In [ ]:
# Starts the worker and blocks, listening for LiveKit to dispatch a room to us.
#
# If this prints "Server stopped or crashed: worker is already running", the
# SDK's in-memory worker registry (Error #10) is still holding a stale
# reference from an earlier run of this cell. That registry lives in the
# kernel's process memory, not in any file, so the fix is a full kernel reset:
#   Kernel -> Restart Kernel, then Run All cells again from the top.
print("Starting agent server to listen for LiveKit Console...")
try:
    await server.run()
except Exception as e:
    print(f"Server stopped or crashed: {e}")
